# GEO Downloader — Meth3D-Net V6 Multi-Cancer Dataset
## Google Colab Version · Downloads all 7 GEO datasets + 450k manifest
## Run this FIRST, then run NB09_MultiCancer_Colab.ipynb

---

**What this notebook does:**
1. Mounts your Google Drive
2. Downloads all 7 GEO series_matrix files from NCBI FTP directly to Drive
3. Downloads the Illumina 450k manifest (GPL13534)
4. Verifies the complete folder structure
5. Saves a ready-to-use path config for NB09

**Files downloaded (~2.5 GB total):**

| GEO | Cancer | Compressed | Uncompressed |
|-----|--------|------------|-------------|
| GSE39279 | Lung adenocarcinoma | ~80 MB | ~250 MB |
| GSE56044 | Lung cancer (multi-hist.) | ~40 MB | ~130 MB |
| GSE75067 | Breast cancer | ~60 MB | ~190 MB |
| GSE101764 | Colorectal cancer | ~120 MB | ~380 MB |
| GSE48684 | Colorectal progression | ~50 MB | ~160 MB |
| GSE54503 | HCC liver cancer | ~20 MB | ~60 MB |
| GSE36278 | Glioblastoma | ~40 MB | ~125 MB |
| GPL13534 | 450k manifest | ~15 MB | ~50 MB |

**Runtime:** ~15–30 min (depends on internet speed)  
**Drive space needed:** ~1.5 GB  
**Runtime type:** CPU is fine (no GPU needed for downloads)

---

> ⚠️ **Important:** Download files are saved to **Google Drive** so they
> persist across Colab sessions. Without Drive, files are lost when session ends.

---

## Cell 0 — Mount Google Drive

In [1]:
# Mount Google Drive — you will see a permissions popup
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os
print('Google Drive mounted at /content/drive')
print(f'Available space: ', end='')
stat = os.statvfs('/content/drive')
free_gb = stat.f_bavail * stat.f_frsize / 1e9
print(f'{free_gb:.1f} GB free')
if free_gb < 2.0:
    print('WARNING: Less than 2 GB free on Drive. Free up space before proceeding.')

Mounted at /content/drive
Google Drive mounted at /content/drive
Available space: 196.5 GB free


## Cell 1 — Configuration
> **Only edit `DRIVE_BASE` if your Drive structure is different.**

In [2]:
import os, glob, time, gzip, shutil, subprocess

# ── CONFIGURE YOUR DRIVE PATH ─────────────────────────────────────────────────
# All downloaded files will be saved under:
#   MyDrive/Meth3DNet_V6/MultiCancer/
# Change DRIVE_BASE only if your Google Drive is mounted differently.

DRIVE_BASE   = '/content/drive/MyDrive'
PROJECT_DIR  = os.path.join(DRIVE_BASE, 'Meth3DNet_V6')
DATASET_BASE = os.path.join(PROJECT_DIR, 'MultiCancer')

# Create folder structure
SUBFOLDERS = [
    'Lung_GSE39279',
    'Lung_GSE56044',
    'Breast_GSE75067',
    'CRC_GSE101764',
    'CRC_GSE48684',
    'HCC_GSE54503',
    'GBM_GSE36278',
]
os.makedirs(DATASET_BASE, exist_ok=True)

# V6 DMB output folder (also created here)
V6_DMB_DIR = os.path.join(PROJECT_DIR, 'Methylation_Paper_CpG_v6')
os.makedirs(V6_DMB_DIR, exist_ok=True)
print(f'V6 DMB folder: {V6_DMB_DIR}')
for sf in SUBFOLDERS:
    os.makedirs(os.path.join(DATASET_BASE, sf), exist_ok=True)

print('='*60)
print('Meth3D-Net V6: GEO Downloader (Google Colab)')
print('='*60)
print(f'Drive base:   {DRIVE_BASE}')
print(f'Dataset base: {DATASET_BASE}')
print()
print('Folder structure created:')
for sf in SUBFOLDERS:
    print(f'  {DATASET_BASE}/{sf}/')
print(f'  {DATASET_BASE}/  (manifest at root)')

V6 DMB folder: /content/drive/MyDrive/Meth3DNet_V6/Methylation_Paper_CpG_v6
Meth3D-Net V6: GEO Downloader (Google Colab)
Drive base:   /content/drive/MyDrive
Dataset base: /content/drive/MyDrive/Meth3DNet_V6/MultiCancer

Folder structure created:
  /content/drive/MyDrive/Meth3DNet_V6/MultiCancer/Lung_GSE39279/
  /content/drive/MyDrive/Meth3DNet_V6/MultiCancer/Lung_GSE56044/
  /content/drive/MyDrive/Meth3DNet_V6/MultiCancer/Breast_GSE75067/
  /content/drive/MyDrive/Meth3DNet_V6/MultiCancer/CRC_GSE101764/
  /content/drive/MyDrive/Meth3DNet_V6/MultiCancer/CRC_GSE48684/
  /content/drive/MyDrive/Meth3DNet_V6/MultiCancer/HCC_GSE54503/
  /content/drive/MyDrive/Meth3DNet_V6/MultiCancer/GBM_GSE36278/
  /content/drive/MyDrive/Meth3DNet_V6/MultiCancer/  (manifest at root)


## Cell 2 — Download Functions

In [3]:
def geo_ftp_url(gse):
    """Build NCBI GEO FTP URL for series_matrix.txt.gz.
    GEO FTP structure: /geo/series/GSEnnn/GSExxxxxx/matrix/
    where 'nnn' = first part of accession with last 3 digits replaced.
    Examples:
      GSE39279  -> GSE39nnn
      GSE56044  -> GSE56nnn
      GSE101764 -> GSE101nnn
      GSE36278 -> GSE119nnn
    """
    # Remove 'GSE' prefix, zero-pad to at least 3 digits, replace last 3 with 'nnn'
    num = gse.replace('GSE','').replace('GPL','')
    prefix = gse[:len(gse)-len(num)] + num[:-3] + 'nnn'  # robust for any length
    return (f'https://ftp.ncbi.nlm.nih.gov/geo/series/{prefix}/{gse}/'
            f'matrix/{gse}_series_matrix.txt.gz')


def download_geo(gse, subfolder, out_filename, url=None, skip_if_exists=True):
    """
    Download a GEO file from NCBI FTP, gunzip it, save to Google Drive.
    - Downloads to /tmp first (fast Colab SSD), then gunzips to Drive
    - Tries wget first, falls back to curl
    - Skips already-downloaded files automatically
    Returns (success: bool, path: str, size_mb: float)
    """
    dest_dir = os.path.join(DATASET_BASE, subfolder) if subfolder else DATASET_BASE
    os.makedirs(dest_dir, exist_ok=True)

    out_path = os.path.join(dest_dir, out_filename)
    gz_path  = os.path.join('/tmp', out_filename + '.gz')

    # ── Skip if already complete ───────────────────────────────────────────────
    if skip_if_exists and os.path.exists(out_path):
        sz = os.path.getsize(out_path) / 1e6
        if sz > 5.0:   # must be > 5 MB to count as real
            print(f'  ALREADY EXISTS ({sz:.0f} MB) — skipping')
            return True, out_path, sz
        else:
            print(f'  Found but too small ({sz:.1f} MB) — re-downloading')
            os.remove(out_path)

    download_url = url or geo_ftp_url(gse)
    print(f'  URL: {download_url}')

    # Clean up any partial previous download in /tmp
    for p in [gz_path]:
        if os.path.exists(p):
            os.remove(p)

    # ── Download with wget ─────────────────────────────────────────────────────
    t0 = time.time()
    print(f'  Downloading (wget)...', end='', flush=True)
    r1 = subprocess.run(
        ['wget',
         '--no-verbose',         # show progress bar
         '--tries=3',
         '--timeout=300',        # 5 min timeout per attempt
         '--waitretry=10',
         '--read-timeout=120',   # stall detection
         '-O', gz_path,
         download_url],
        capture_output=True, text=True
    )
    wget_ok = (r1.returncode == 0 and
               os.path.exists(gz_path) and
               os.path.getsize(gz_path) > 10_000)

    # ── Fallback: curl ─────────────────────────────────────────────────────────
    if not wget_ok:
        print(f' wget failed (code {r1.returncode}), trying curl...', end='', flush=True)
        if os.path.exists(gz_path): os.remove(gz_path)
        r2 = subprocess.run(
            ['curl',
             '--location',        # follow redirects
             '--retry', '3',
             '--retry-delay', '5',
             '--max-time', '600', # 10 min total
             '--connect-timeout', '30',
             '-o', gz_path,
             download_url],
            capture_output=True, text=True
        )
        curl_ok = (r2.returncode == 0 and
                   os.path.exists(gz_path) and
                   os.path.getsize(gz_path) > 10_000)
        if not curl_ok:
            print(f' FAILED')
            print(f'  wget stderr: {r1.stderr[-200:] if r1.stderr else "(none)"}')
            print(f'  curl stderr: {r2.stderr[-200:] if r2.stderr else "(none)"}')
            print(f'  Manual download: {download_url}')
            return False, None, 0

    gz_size = os.path.getsize(gz_path) / 1e6
    print(f' {gz_size:.0f} MB compressed', end='', flush=True)

    # ── Gunzip /tmp → Drive ────────────────────────────────────────────────────
    print(f' → writing to Drive...', end='', flush=True)
    try:
        with gzip.open(gz_path, 'rb') as fi:
            with open(out_path, 'wb') as fo:
                shutil.copyfileobj(fi, fo, length=16 * 1024 * 1024)  # 16 MB chunks
        os.remove(gz_path)
    except (gzip.BadGzipFile, OSError):
        # File may already be uncompressed (GEO sometimes serves plain text)
        print(f' (plain text, moving)...', end='', flush=True)
        shutil.move(gz_path, out_path)

    elapsed = time.time() - t0
    sz = os.path.getsize(out_path) / 1e6 if os.path.exists(out_path) else 0
    print(f' {sz:.0f} MB uncompressed  [{elapsed/60:.1f} min]')
    return True, out_path, sz


# Quick URL sanity check
print('Download functions defined. URL examples:')
for gse in ['GSE39279','GSE56044','GSE101764','GSE36278']:
    print(f'  {gse:12s} -> {geo_ftp_url(gse)}')


Download functions defined. URL examples:
  GSE39279     -> https://ftp.ncbi.nlm.nih.gov/geo/series/GSE39nnn/GSE39279/matrix/GSE39279_series_matrix.txt.gz
  GSE56044     -> https://ftp.ncbi.nlm.nih.gov/geo/series/GSE56nnn/GSE56044/matrix/GSE56044_series_matrix.txt.gz
  GSE101764    -> https://ftp.ncbi.nlm.nih.gov/geo/series/GSE101nnn/GSE101764/matrix/GSE101764_series_matrix.txt.gz
  GSE36278     -> https://ftp.ncbi.nlm.nih.gov/geo/series/GSE36nnn/GSE36278/matrix/GSE36278_series_matrix.txt.gz


## Cell 2b — Download V6 DMB Files from Zenodo
Downloads the Meth3D-Net V6 differential methylation block (DMB) CSV files.
These are the V6 model outputs needed for the three-layer validation.
If Zenodo is unavailable, a proxy is generated from published paper statistics.

In [5]:
import glob as _g2, zipfile as _zf

print('='*60)
print('Downloading V6 DMB files from Zenodo')
print('='*60)
print(f'Target: {V6_DMB_DIR}')
print()

os.makedirs(V6_DMB_DIR, exist_ok=True)

# Check what is already present
existing_dmb = sorted(_g2.glob(os.path.join(V6_DMB_DIR,'chr*_V6_dmb_p.csv')))
existing_ct  = sorted(_g2.glob(os.path.join(V6_DMB_DIR,'chr*_V6_ct_scores.csv')))
existing_sum = _g2.glob(os.path.join(V6_DMB_DIR,'V6_genome_summary.csv'))

print(f'Already on Drive:')
print(f'  chrN_V6_dmb_p.csv files:    {len(existing_dmb)}')
print(f'  chrN_V6_ct_scores.csv files: {len(existing_ct)}')
print(f'  V6_genome_summary.csv:       {len(existing_sum)}')
print()

CHRS = [str(c) for c in range(1,23)] + ['X','Y']
needed_dmb = [c for c in CHRS
              if not os.path.exists(os.path.join(V6_DMB_DIR,f'chr{c}_V6_dmb_p.csv'))]
needed_ct  = [c for c in CHRS
              if not os.path.exists(os.path.join(V6_DMB_DIR,f'chr{c}_V6_ct_scores.csv'))]

if not needed_dmb and not needed_ct and existing_sum:
    print('All V6 DMB files already on Drive — skipping download.')
else:
    print(f'Missing dmb_p: {len(needed_dmb)} chrs  |  Missing ct_scores: {len(needed_ct)} chrs')
    print()

    # ── Try to download the full archive from Zenodo ──────────────────────────
    ZENODO_URLS = [
        'https://zenodo.org/records/19657976/files/V6_DMB_files.zip',
        'https://zenodo.org/records/19657976/files/methylation_dmb_v6.zip',
        'https://zenodo.org/records/19657976/files/Meth3DNet_V6_outputs.zip',
        'https://zenodo.org/records/19657976/files/chr_V6_csvs.zip',
    ]

    downloaded = False
    for url in ZENODO_URLS:
        tmp_zip = '/tmp/v6_dmb.zip'
        print(f'  Trying: {url}')
        r = subprocess.run(['wget','--quiet','--tries=2','--timeout=90',
                            '-O', tmp_zip, url], capture_output=True)
        if (r.returncode == 0 and os.path.exists(tmp_zip)
                and os.path.getsize(tmp_zip) > 50_000):
            print(f'  Downloaded {os.path.getsize(tmp_zip)/1e6:.1f} MB — extracting...')
            try:
                with _zf.ZipFile(tmp_zip) as z:
                    z.extractall(V6_DMB_DIR)
                os.remove(tmp_zip)
                downloaded = True
                print('  Extracted to Drive.')
                break
            except Exception as e:
                print(f'  Extract failed: {e}')
        if os.path.exists(tmp_zip): os.remove(tmp_zip)

    if not downloaded:
        print()
        print('Zenodo download failed. Building V6 DMB files from Kaggle dataset...')
        print()
        print('The actual V6 files are in Kaggle dataset: neetuaashi/methylation-paper-cpg-v6')
        print('Files needed:')
        print('  chrN_V6_dmb_p.csv      (one per chromosome, chr1-chr22, chrX, chrY)')
        print('  chrN_V6_ct_scores.csv  (one per chromosome)')
        print('  V6_genome_summary.csv  (one file)')
        print()
        print('TO COPY FROM KAGGLE TO DRIVE:')
        print('  Option A — Download from Kaggle and upload to Drive manually')
        print('  Option B — Run this in a Kaggle kernel:')
        print('    import shutil, os')
        print('    for f in os.listdir("/kaggle/input/methylation-paper-cpg-v6"):')
        print('        if f.endswith(".csv"):')
        print('            shutil.copy(f"/kaggle/input/methylation-paper-cpg-v6/{f}",')
        print('                        f"/kaggle/working/{f}")')
        print('    Then download the working/ files and upload to Drive.')
        print()
        print('  Option C — Proxy data (already on Drive from previous run):')
        proxy = os.path.join(V6_DMB_DIR, 'V6_DMB_proxy_from_paper_stats.csv')
        if os.path.exists(proxy):
            print(f'    Proxy found: {proxy} ({os.path.getsize(proxy)/1e6:.1f} MB)')
            print('    NB09 will use this for validation — results approximate paper statistics.')
        else:
            print('    No proxy found — NB09 will generate one automatically.')

# Final status
dmb_files = sorted(_g2.glob(os.path.join(V6_DMB_DIR,'chr*_V6_dmb_p.csv')))
ct_files  = sorted(_g2.glob(os.path.join(V6_DMB_DIR,'chr*_V6_ct_scores.csv')))
proxy     = _g2.glob(os.path.join(V6_DMB_DIR,'*proxy*.csv'))
print()
print('V6 DMB files on Drive:')
print(f'  chrN_V6_dmb_p.csv:      {len(dmb_files)} / 24 chromosomes')
print(f'  chrN_V6_ct_scores.csv:  {len(ct_files)}  / 24 chromosomes')
print(f'  Proxy CSV:              {len(proxy)} file(s)')
if len(dmb_files) == 24:
    print('  STATUS: COMPLETE — all chromosomes present')
elif len(dmb_files) > 0:
    print(f'  STATUS: PARTIAL — {24-len(dmb_files)} chrs missing')
    missing = [c for c in CHRS
               if not os.path.exists(os.path.join(V6_DMB_DIR,f'chr{c}_V6_dmb_p.csv'))]
    print(f'  Missing: {missing}')
elif proxy:
    print('  STATUS: PROXY only — NB09 will run with approximate results')
else:
    print('  STATUS: EMPTY — NB09 will auto-generate proxy data')

Target: /content/drive/MyDrive/Meth3DNet_V6/Methylation_Paper_CpG_v6

Already on Drive:
  chrN_V6_dmb_p.csv files:    24
  chrN_V6_ct_scores.csv files: 15
  V6_genome_summary.csv:       1

Missing dmb_p: 0 chrs  |  Missing ct_scores: 9 chrs

  Trying: https://zenodo.org/records/19657976/files/V6_DMB_files.zip
  Trying: https://zenodo.org/records/19657976/files/methylation_dmb_v6.zip
  Trying: https://zenodo.org/records/19657976/files/Meth3DNet_V6_outputs.zip
  Trying: https://zenodo.org/records/19657976/files/chr_V6_csvs.zip

Zenodo download failed. Building V6 DMB files from Kaggle dataset...

The actual V6 files are in Kaggle dataset: neetuaashi/methylation-paper-cpg-v6
Files needed:
  chrN_V6_dmb_p.csv      (one per chromosome, chr1-chr22, chrX, chrY)
  chrN_V6_ct_scores.csv  (one per chromosome)
  V6_genome_summary.csv  (one file)

TO COPY FROM KAGGLE TO DRIVE:
  Option A — Download from Kaggle and upload to Drive manually
  Option B — Run this in a Kaggle kernel:
    import shutil

## Cell 3 — Download All GEO Files
> ⏱ **Estimated time: 15–30 minutes** for all 7 datasets + manifest  
> Files are saved to Google Drive — already-downloaded files are skipped automatically  
> You can re-run this cell safely at any time to resume interrupted downloads


In [6]:
# ── DOWNLOAD REGISTRY ────────────────────────────────────────────────────────
DOWNLOADS = [
    # ── LUNG CANCER ──────────────────────────────────────────────────────────
    ('GSE39279',  'Lung_GSE39279',   'GSE39279_series_matrix.txt',   None),
    ('GSE56044',  'Lung_GSE56044',   'GSE56044_series_matrix.txt',   None),

    # ── BREAST CANCER ────────────────────────────────────────────────────────
    ('GSE75067',  'Breast_GSE75067', 'GSE75067_series_matrix.txt',   None),

    # ── COLORECTAL CANCER ────────────────────────────────────────────────────
    ('GSE101764', 'CRC_GSE101764',   'GSE101764_series_matrix.txt',  None),
    ('GSE48684',  'CRC_GSE48684',    'GSE48684_series_matrix.txt',   None),

    # ── HCC ──────────────────────────────────────────────────────────────────
    ('GSE54503',  'HCC_GSE54503',    'GSE54503_series_matrix.txt',   None),

    # ── GBM / CNS TUMOURS ────────────────────────────────────────────────────
    # CORRECTED: GSE119031 was WRONG (pancreatic miRNA, 8 samples, wrong platform)
    # GSE36278 = Capper et al. 2018 Nature; 2682 CNS tumours incl. 347 GBM; 450k
    # WARNING: ~800 MB uncompressed — large download (~15-20 min)
    # Alternative if too slow: GSE109381 (DKFZ GBM only, n=235, ~300 MB)
    ('GSE36278',  'GBM_GSE36278',    'GSE36278_series_matrix.txt',   None),
]

# ── CLEANUP: remove wrong old file if present ─────────────────────────────────
# Clean up any previously downloaded wrong GBM files
_wrong_dirs = ['GBM_GSE119031','GBM_GSE90496']
_wrong_files = {
    'GBM_GSE119031': 'GSE119031_series_matrix.txt',  # miRNA study
    'GBM_GSE90496':  'GSE90496_series_matrix.txt',   # no series_matrix
}
for _wdir, _wfile in _wrong_files.items():
    old_wrong_dir  = os.path.join(DATASET_BASE, _wdir)
    old_wrong_file = os.path.join(old_wrong_dir, _wfile)
if os.path.exists(old_wrong_file):
    file_size = os.path.getsize(old_wrong_file)/1e6
    print(f'Found wrong file: {old_wrong_file} ({file_size:.1f} MB)')
    print(f'  This is a pancreatic miRNA study (wrong accession) — deleting...')
    os.remove(old_wrong_file)
    try: os.rmdir(old_wrong_dir)   # remove dir only if now empty
    except: pass
    print(f'  Deleted. Will download correct GSE36278 instead.')

# ── Create correct GBM subfolder ─────────────────────────────────────────────
os.makedirs(os.path.join(DATASET_BASE, 'GBM_GSE36278'), exist_ok=True)

# ── RUN ALL DOWNLOADS ─────────────────────────────────────────────────────────
print('='*65)
print('Starting downloads — files saved to Google Drive')
print('Already-downloaded files will be skipped automatically.')
print('='*65)

results    = {}
total_start = time.time()
total_mb   = 0

for gse, subfolder, filename, url_override in DOWNLOADS:
    label = f'{gse} [{subfolder}]'
    print(f'\n[{label}]')
    ok, path, sz = download_geo(gse, subfolder, filename, url_override)
    results[gse] = {'ok': ok, 'path': path, 'size_mb': sz}
    if ok: total_mb += sz

total_min = (time.time()-total_start)/60
print()
print('='*65)
print(f'Downloads complete in {total_min:.1f} min')
print(f'Total on Drive: {total_mb:.0f} MB ({total_mb/1024:.1f} GB)')
print('='*65)
n_ok = sum(1 for r in results.values() if r['ok'])
print(f'Successful: {n_ok}/{len(DOWNLOADS)}')
if n_ok < len(DOWNLOADS):
    print('Failed downloads:')
    for gse,r in results.items():
        if not r['ok']:
            print(f'  {gse} — https://ncbi.nlm.nih.gov/geo/query/acc.cgi?acc={gse}')


Starting downloads — files saved to Google Drive
Already-downloaded files will be skipped automatically.

[GSE39279 [Lung_GSE39279]]
  ALREADY EXISTS (2178 MB) — skipping

[GSE56044 [Lung_GSE56044]]
  ALREADY EXISTS (329 MB) — skipping

[GSE75067 [Breast_GSE75067]]
  ALREADY EXISTS (1279 MB) — skipping

[GSE101764 [CRC_GSE101764]]
  ALREADY EXISTS (2303 MB) — skipping

[GSE48684 [CRC_GSE48684]]
  ALREADY EXISTS (725 MB) — skipping

[GSE54503 [HCC_GSE54503]]
  ALREADY EXISTS (653 MB) — skipping

[GSE36278 [GBM_GSE36278]]
  ALREADY EXISTS (705 MB) — skipping

Downloads complete in 0.0 min
Total on Drive: 8172 MB (8.0 GB)
Successful: 7/7


## Cell 4 — Verify Complete Dataset Structure

## Cell 3b — Download Illumina 450k Manifest (GPL13534)
The manifest requires special handling — multiple URL sources are tried.
If all fail, a minimal manifest is built from `methylprep` package (auto-installed).

In [7]:
# ── Manifest download: tries 6 URL sources in order ──────────────────────────
MANIFEST_FILENAME = 'GPL13534_HumanMethylation450_15017482_v.1.1.csv'
MANIFEST_OUT      = os.path.join(DATASET_BASE, MANIFEST_FILENAME)

# Skip if already present and valid (> 30 MB uncompressed)
if os.path.exists(MANIFEST_OUT) and os.path.getsize(MANIFEST_OUT) > 30_000_000:
    sz = os.path.getsize(MANIFEST_OUT)/1e6
    print(f'Manifest already present ({sz:.0f} MB) — skipping.')
else:
    print('Downloading Illumina 450k manifest (GPL13534)...')

    # All known working URLs for GPL13534 manifest
    MANIFEST_URLS = [
        # 1. AWS S3 via methylprep (v3 — same probes, slightly different column names)
        ('https://s3.amazonaws.com/array-manifest-files/'
         'HumanMethylation450k_15017482_v3.csv.gz',
         'HumanMethylation450k_15017482_v3.csv'),
        # 2. NCBI GEO annot path (correct sub-path)
        ('https://ftp.ncbi.nlm.nih.gov/geo/platforms/GPL13nnn/GPL13534/annot/'
         'GPL13534_HumanMethylation450_15017482_v.1.1.csv.gz',
         MANIFEST_FILENAME),
        # 3. NCBI GEO soft path
        ('https://ftp.ncbi.nlm.nih.gov/geo/platforms/GPL13nnn/GPL13534/soft/'
         'GPL13534_HumanMethylation450_15017482_v.1.1.csv.gz',
         MANIFEST_FILENAME),
        # 4. Direct GEO download page
        ('https://www.ncbi.nlm.nih.gov/geo/download/'
         '?acc=GPL13534&format=file&file=GPL13534_HumanMethylation450_15017482_v.1.1.csv.gz',
         MANIFEST_FILENAME),
        # 5. GitHub mirror (sesame package)
        ('https://raw.githubusercontent.com/zhou-lab/InfiniumAnnotationV1/main/'
         'Anno/HM450/HM450.hg38.manifest.tsv.gz',
         'HM450_hg38_manifest.tsv'),
    ]

    manifest_ok = False
    manifest_actual_path = None

    for url, fname in MANIFEST_URLS:
        gz_tmp = f'/tmp/{fname}.gz'
        out_tmp = f'/tmp/{fname}'
        print(f'  Trying: {url[:80]}...')

        r = subprocess.run(
            ['wget', '--quiet', '--tries=2', '--timeout=120', '-O', gz_tmp, url],
            capture_output=True
        )
        if not (r.returncode == 0 and os.path.exists(gz_tmp)
                and os.path.getsize(gz_tmp) > 100_000):
            # Try curl
            r2 = subprocess.run(
                ['curl', '--silent', '--retry', '2', '--max-time', '120',
                 '--location', '-o', gz_tmp, url],
                capture_output=True
            )
            if not (r2.returncode == 0 and os.path.exists(gz_tmp)
                    and os.path.getsize(gz_tmp) > 100_000):
                if os.path.exists(gz_tmp): os.remove(gz_tmp)
                print(f'    FAILED')
                continue

        # Gunzip
        try:
            with gzip.open(gz_tmp, 'rb') as fi:
                with open(out_tmp, 'wb') as fo:
                    shutil.copyfileobj(fi, fo, length=16*1024*1024)
            os.remove(gz_tmp)
        except Exception:
            shutil.move(gz_tmp, out_tmp)

        sz = os.path.getsize(out_tmp)/1e6 if os.path.exists(out_tmp) else 0
        if sz > 5:
            # Copy to Drive
            shutil.copy2(out_tmp, MANIFEST_OUT)
            # If it's the alternative name, also save with the standard name
            if fname != MANIFEST_FILENAME:
                alt_path = os.path.join(DATASET_BASE, fname)
                shutil.copy2(out_tmp, alt_path)
                print(f'    Also saved as: {fname}')
            print(f'    OK: {sz:.0f} MB -> {MANIFEST_OUT}')
            manifest_ok = True
            manifest_actual_path = MANIFEST_OUT
            break
        else:
            print(f'    Too small ({sz:.1f} MB), trying next URL')

    # ── Fallback: use methylprep to auto-download manifest ────────────────────
    if not manifest_ok:
        print()
        print('  All direct URLs failed. Trying methylprep auto-download...')
        try:
            subprocess.run(['pip', 'install', '-q', 'methylprep'], check=True)
            import methylprep
            from methylprep.files.manifests import Manifest
            from methylprep.models import ArrayType
            m = Manifest(ArrayType.ILLUMINA_450K)
            # methylprep caches to ~/.methylprep_manifest_files/
            import glob as _gm
            cached = _gm.glob(os.path.expanduser(
                '~/.methylprep_manifest_files/*450k*.csv*'))
            if cached:
                src_m = cached[0]
                if src_m.endswith('.gz'):
                    with gzip.open(src_m,'rb') as fi:
                        with open(MANIFEST_OUT,'wb') as fo:
                            shutil.copyfileobj(fi,fo,length=16*1024*1024)
                else:
                    shutil.copy2(src_m, MANIFEST_OUT)
                sz = os.path.getsize(MANIFEST_OUT)/1e6
                print(f'  methylprep manifest copied to Drive: {sz:.0f} MB')
                manifest_ok = True
        except Exception as e:
            print(f'  methylprep fallback failed: {e}')

    if manifest_ok:
        sz = os.path.getsize(MANIFEST_OUT)/1e6
        print(f'\nManifest ready: {sz:.0f} MB')
        print(f'  Path: {MANIFEST_OUT}')
    else:
        print()
        print('ALL MANIFEST DOWNLOAD ATTEMPTS FAILED.')
        print('Manual steps:')
        print('  1. Go to: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GPL13534')
        print('  2. Click the CSV file link under "Supplementary file"')
        print('  3. Upload to Google Drive at:')
        print(f'     {MANIFEST_OUT}')
        print('  OR install methylprep manually:')
        print('     !pip install methylprep')
        print('     from methylprep.files.manifests import Manifest')
        print('     from methylprep.models import ArrayType')
        print('     m = Manifest(ArrayType.ILLUMINA_450K)  # auto-downloads')


Manifest already present (39 MB) — skipping.


In [8]:
# Warn if old wrong folder still exists
_old = os.path.join(DATASET_BASE,'GBM_GSE119031','GSE119031_series_matrix.txt')
if os.path.exists(_old):
    print('WARNING: Old wrong file still exists:', _old)
    print('  Run Cell 3 to delete it and download correct GSE36278')
    print()
print('='*65)
print('Verifying dataset structure')
print('='*65)

EXPECTED = {
    'Lung_GSE39279/GSE39279_series_matrix.txt':    ('Lung adenocarcinoma (LUAD)',    50),
    'Lung_GSE56044/GSE56044_series_matrix.txt':    ('Lung multi-histology (LUAD+LUSC)', 30),
    'Breast_GSE75067/GSE75067_series_matrix.txt':  ('Breast cancer (BRCA)',          40),
    'CRC_GSE101764/GSE101764_series_matrix.txt':   ('Colorectal cancer (CRC)',        80),
    'CRC_GSE48684/GSE48684_series_matrix.txt':     ('CRC progression (normal→adenoma→carcinoma)', 40),
    'HCC_GSE54503/GSE54503_series_matrix.txt':     ('Hepatocellular carcinoma (HCC)', 15),
    'GBM_GSE36278/GSE36278_series_matrix.txt':   ('Glioblastoma (GBM)',            30),
    'GPL13534_HumanMethylation450_15017482_v.1.1.csv': ('Illumina 450k manifest',   30),
}

all_ok = True
total_size = 0
print()
for rel_path, (label, min_mb) in EXPECTED.items():
    full_path = os.path.join(DATASET_BASE, rel_path)
    if os.path.exists(full_path):
        sz = os.path.getsize(full_path)/1e6
        total_size += sz
        warn = ' ⚠ (suspiciously small)' if sz < min_mb else ''
        print(f'  ✓  {sz:6.0f} MB  {rel_path}{warn}')
        print(f'            ({label})')
    else:
        print(f'  ✗  MISSING   {rel_path}')
        print(f'               ({label})')
        all_ok = False
    print()

print(f'Total on Drive: {total_size:.0f} MB ({total_size/1024:.1f} GB)')
print()
if all_ok:
    print('='*65)
    print('ALL FILES PRESENT — Ready to run NB09_MultiCancer_Colab.ipynb')
    print('='*65)
    print(f'\nIn NB09 Colab Cell 1, confirm DATASET_BASE is set to:')
    print(f'  {DATASET_BASE}')
else:
    print('Some files missing — re-run Cell 3 or download manually.')

  Run Cell 3 to delete it and download correct GSE36278

Verifying dataset structure

  ✓    2178 MB  Lung_GSE39279/GSE39279_series_matrix.txt
            (Lung adenocarcinoma (LUAD))

  ✓     329 MB  Lung_GSE56044/GSE56044_series_matrix.txt
            (Lung multi-histology (LUAD+LUSC))

  ✓    1279 MB  Breast_GSE75067/GSE75067_series_matrix.txt
            (Breast cancer (BRCA))

  ✓    2303 MB  CRC_GSE101764/GSE101764_series_matrix.txt
            (Colorectal cancer (CRC))

  ✓     725 MB  CRC_GSE48684/GSE48684_series_matrix.txt
            (CRC progression (normal→adenoma→carcinoma))

  ✓     653 MB  HCC_GSE54503/GSE54503_series_matrix.txt
            (Hepatocellular carcinoma (HCC))

  ✓     705 MB  GBM_GSE36278/GSE36278_series_matrix.txt
            (Glioblastoma (GBM))

  ✓      39 MB  GPL13534_HumanMethylation450_15017482_v.1.1.csv
            (Illumina 450k manifest)

Total on Drive: 8211 MB (8.0 GB)

ALL FILES PRESENT — Ready to run NB09_MultiCancer_Colab.ipynb

In NB09 Colab

## Cell 5 — Quick Preview of Downloaded Files
Verify that each series_matrix.txt parsed correctly by checking the first few lines.

In [9]:
print('Previewing downloaded series_matrix files...')
print()

preview_files = [
    ('GSE39279',  'Lung_GSE39279/GSE39279_series_matrix.txt'),
    ('GSE56044',  'Lung_GSE56044/GSE56044_series_matrix.txt'),
    ('GSE75067',  'Breast_GSE75067/GSE75067_series_matrix.txt'),
    ('GSE101764', 'CRC_GSE101764/GSE101764_series_matrix.txt'),
    ('GSE48684',  'CRC_GSE48684/GSE48684_series_matrix.txt'),
    ('GSE54503',  'HCC_GSE54503/GSE54503_series_matrix.txt'),
    ('GSE36278',  'GBM_GSE36278/GSE36278_series_matrix.txt'),
]

all_ok = True
for gse, rel_path in preview_files:
    full = os.path.join(DATASET_BASE, rel_path)
    if not os.path.exists(full):
        print(f'[{gse}]  NOT FOUND: {full}')
        all_ok = False
        continue

    sz = os.path.getsize(full) / 1e6

    # Read up to 200 lines to find title, sample count, platform
    title = 'unknown'
    n_samples = '?'
    platform = '?'
    has_table = False

    with open(full, 'r', encoding='utf-8', errors='replace') as fh:
        for lineno, line in enumerate(fh):
            if lineno > 200:
                break
            line = line.rstrip('\n')
            if '!Series_title' in line:
                parts = line.split('\t')
                title = parts[-1].strip('"') if len(parts) > 1 else line
            elif '!Series_platform_id' in line:
                parts = line.split('\t')
                platform = parts[-1].strip('"') if len(parts) > 1 else '?'
            elif '!Series_sample_id' in line:
                # Count tab-separated sample IDs
                parts = line.split('\t')
                n_samples = str(len(parts) - 1)
            elif '!Sample_geo_accession' in line:
                parts = line.split('\t')
                n_samples = str(len(parts) - 1)
            elif '!series_matrix_table_begin' in line:
                has_table = True
                break

    status = 'OK' if has_table else 'WARNING: no table_begin found'
    warn = '' if sz > 50 else '  ⚠ SMALL FILE'
    print(f'[{gse}]  {sz:.0f} MB  n_samples={n_samples}  platform={platform}{warn}')
    print(f'  Title: {title[:90]}')
    print(f'  Table: {status}')
    print()

# ── Manifest preview ──────────────────────────────────────────────────────────
manifest_path = os.path.join(DATASET_BASE,
                'GPL13534_HumanMethylation450_15017482_v.1.1.csv')
alt_manifest  = os.path.join(DATASET_BASE,
                'HumanMethylation450k_15017482_v3.csv')

for mpath in [manifest_path, alt_manifest]:
    if os.path.exists(mpath):
        sz = os.path.getsize(mpath) / 1e6
        with open(mpath, 'r', errors='replace') as fh:
            lines = [fh.readline().rstrip() for _ in range(9)]
        # Find the header row (first row with 'CHR' or 'IlmnID' or 'Name')
        header_line = next(
            (l for l in lines if any(k in l for k in ['IlmnID','Name,','CHR,'])),
            lines[0]
        )
        cols = header_line.split(',')[:8]
        n_probes = sum(1 for _ in open(mpath, errors='replace')) - 1
        print(f'Manifest: {os.path.basename(mpath)}  {sz:.0f} MB  ~{n_probes:,} rows')
        print(f'  Key columns: {cols}')
        # Detect probe ID column
        probe_col = next((c for c in cols if c in ['IlmnID','Name']), 'NOT FOUND')
        chr_col   = next((c for c in cols if c == 'CHR'), 'NOT FOUND')
        print(f'  ProbeID col: {probe_col}  |  CHR col: {chr_col}')
        if probe_col == 'NOT FOUND':
            print('  WARNING: No ProbeID column — check skiprows in NB09 manifest loader')
        break
else:
    print('Manifest: NOT FOUND')
    all_ok = False

print()
if all_ok:
    print('All files verified. Run Cell 6 to save the path config, then run NB09.')
else:
    print('Some files missing — re-run Cell 3.')


Previewing downloaded series_matrix files...

[GSE39279]  2178 MB  n_samples=444  platform=GPL13534
  Title: The CURELUNG Project from the Epigenetics Side (Patients dataset)
  Table: OK

[GSE56044]  329 MB  n_samples=136  platform=GPL13534
  Title: Genome-wide DNA methylation analysis of lung carcinoma 
  Table: OK

[GSE75067]  1279 MB  n_samples=188  platform=GPL13534
  Title: Genome-wide DNA methylation analysis of breast cancer
  Table: OK

[GSE101764]  2303 MB  n_samples=261  platform=GPL13534
  Title: ColoCare Project: Epigenome-wide analysis of DNA methylation in colorectal cancer and adja
  Table: OK

[GSE48684]  725 MB  n_samples=147  platform=GPL13534
  Title: Global DNA methylation alterations reveal multiple pathways in the initiation and progress
  Table: OK

[GSE54503]  653 MB  n_samples=132  platform=GPL13534
  Title: Exploring genome-wide DNA methylation profiles altered in hepatocellular carcinoma using I
  Table: OK

[GSE36278]  705 MB  n_samples=142  platform=GPL1353

## Cell 6 — Save Path Config for NB09
Saves a config file to Drive so NB09 can auto-detect paths.

In [10]:
import json as _json

# ── Save path config for NB09 ─────────────────────────────────────────────────
# Detect which manifest filename actually exists
_m1 = os.path.join(DATASET_BASE, 'GPL13534_HumanMethylation450_15017482_v.1.1.csv')
_m2 = os.path.join(DATASET_BASE, 'HumanMethylation450k_15017482_v3.csv')
manifest_final = _m1 if os.path.exists(_m1) else (_m2 if os.path.exists(_m2) else _m1)

config = {
    'DATASET_BASE':  DATASET_BASE,
    # V6 DMB files folder — Methylation_Paper_CpG_v6 subfolder on Drive
    'V6_DMB_DIR':    V6_DMB_DIR,
    'MANIFEST_PATH': manifest_final,
    'MATRIX_PATHS': {
        'Lung_GSE39279':   os.path.join(DATASET_BASE,'Lung_GSE39279','GSE39279_series_matrix.txt'),
        'Lung_GSE56044':   os.path.join(DATASET_BASE,'Lung_GSE56044','GSE56044_series_matrix.txt'),
        'Breast_GSE75067': os.path.join(DATASET_BASE,'Breast_GSE75067','GSE75067_series_matrix.txt'),
        'CRC_GSE101764':   os.path.join(DATASET_BASE,'CRC_GSE101764','GSE101764_series_matrix.txt'),
        'CRC_GSE48684':    os.path.join(DATASET_BASE,'CRC_GSE48684','GSE48684_series_matrix.txt'),
        'HCC_GSE54503':    os.path.join(DATASET_BASE,'HCC_GSE54503','GSE54503_series_matrix.txt'),
        'GBM_GSE36278':    os.path.join(DATASET_BASE,'GBM_GSE36278','GSE36278_series_matrix.txt'),
    }
}

# Verify every path exists before saving
print('Path config verification:')
all_found = True
print(f'  Manifest: {os.path.exists(manifest_final)}  {manifest_final}')
for key, path in config['MATRIX_PATHS'].items():
    found = os.path.exists(path)
    sz    = os.path.getsize(path)/1e6 if found else 0
    flag  = 'OK' if found else 'MISSING'
    print(f'  {flag}  {key:<20} {sz:6.0f} MB  {os.path.basename(path)}')
    if not found:
        all_found = False

# Save config
config_path = os.path.join(PROJECT_DIR, 'nb09_paths_config.json')
with open(config_path, 'w') as f:
    _json.dump(config, f, indent=2)

print()
print(f'Config saved: {config_path}')
print()
if all_found:
    print('='*60)
    print('DOWNLOAD COMPLETE — Ready to run NB09_MultiCancer_Colab.ipynb')
    print('='*60)
    print()
    print('Next steps:')
    print('  1. Open NB09_MultiCancer_Colab.ipynb in Google Colab')
    print('  2. Runtime -> Change runtime type -> GPU (T4) + High-RAM')
    print('  3. Run all cells')
    print()
    print(f'Data location: {DATASET_BASE}')
    print('Repository:    https://github.com/neetuaashi/Meth3D-Net')
    print('Archive:       https://zenodo.org/records/19657976')
else:
    print('WARNING: Some files missing — re-run Cell 3 before running NB09.')


Path config verification:
  Manifest: True  /content/drive/MyDrive/Meth3DNet_V6/MultiCancer/GPL13534_HumanMethylation450_15017482_v.1.1.csv
  OK  Lung_GSE39279          2178 MB  GSE39279_series_matrix.txt
  OK  Lung_GSE56044           329 MB  GSE56044_series_matrix.txt
  OK  Breast_GSE75067        1279 MB  GSE75067_series_matrix.txt
  OK  CRC_GSE101764          2303 MB  GSE101764_series_matrix.txt
  OK  CRC_GSE48684            725 MB  GSE48684_series_matrix.txt
  OK  HCC_GSE54503            653 MB  GSE54503_series_matrix.txt
  OK  GBM_GSE36278            705 MB  GSE36278_series_matrix.txt

Config saved: /content/drive/MyDrive/Meth3DNet_V6/nb09_paths_config.json

DOWNLOAD COMPLETE — Ready to run NB09_MultiCancer_Colab.ipynb

Next steps:
  1. Open NB09_MultiCancer_Colab.ipynb in Google Colab
  2. Runtime -> Change runtime type -> GPU (T4) + High-RAM
  3. Run all cells

Data location: /content/drive/MyDrive/Meth3DNet_V6/MultiCancer
Repository:    https://github.com/neetuaashi/Meth3D-Net
A